# Lesson 1.3 — coordinate frame 与坐标变换

dataset 中的每一个 pose 都是**相对于某个东西**表达的。本 notebook 将这一点显式化，然后验证 policy 接收到的相对向量。

这里演示而不是描述两个陷阱：

1. `Pose.q` 中的 **quaternion 分量顺序**；
2. 一个**减法顺序错误**，它会静默地产生一个看起来合理但实际错误的相对向量。

## 1.3.1 — 前置准备

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## 1.3.2 — `Pose` API：位置与姿态

一个 7 个数的 pose 是 3 个平移值加一个 4 个数的 quaternion。是哪四个数、按什么顺序至关重要 —— 弄错了，姿态就毫无意义，而平移看起来仍然正确。

In [2]:
env = make_env()
agent = env.unwrapped.agent

pose = agent.tcp_pose
print("Pose object :", pose)
print("raw (7 values):", pose.raw_pose[0])
print("p (position)  :", pose.p[0])
print("q (quaternion):", pose.q[0])
print("\nThe same 7 values appear in the dataset as tcp_pose.")

2026-09-22 11:14:18,291 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


Pose object : Pose(raw_pose=tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]]))
raw (7 values): tensor([ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080])
p (position)  : tensor([0.0123, 0.0380, 0.1822])
q (quaternion): tensor([-0.0177,  0.9998,  0.0043,  0.0080])

The same 7 values appear in the dataset as tcp_pose.


## 1.3.3 — 陷阱：`Pose.q` 中的 quaternion 顺序

`Pose.q` 是 **(x, y, z, w)** —— scipy 约定。SAPIEN 文档与许多机器人代码库使用 **(w, x, y, z)**。两者只是分量顺序的旋转不同，因此都"看起来像"一个合法的 quaternion。

下面的测试在两种假设下转换同一组数字。当 TCP 朝下时，正确的解释得到的 rotation 接近一个干净的单轴 rotation；错误的解释则产生绕意外轴的大角度。

> 下面的数值是在一次实际会话中观察到的：正确的 (x, y, z, w) 读法给出约 `[3.13, 0.016, -3.11]` 弧度，即近似绕 z 的纯 rotation，而 (w, x, y, z) 读法给出约
> `[-0.035, -0.016, 3.13]`。

In [3]:
from scipy.spatial.transform import Rotation as R

quat = pose.q[0].cpu().numpy()
print("raw quaternion values:", quat)

as_xyzw = R.from_quat(quat).as_euler("xyz")
as_wxyz = R.from_quat(np.roll(quat, 1)).as_euler("xyz")

print("\nif (x, y, z, w):", np.round(as_xyzw, 4), "rad ->", np.round(np.rad2deg(as_xyzw), 2), "deg")
print("if (w, x, y, z):", np.round(as_wxyz, 4), "rad ->", np.round(np.rad2deg(as_wxyz), 2), "deg")
print("\nPose.q is (x, y, z, w): the first reading is a near-pure z rotation.")

raw quaternion values: [-0.01768896  0.9998025   0.00428796  0.00798423]

if (x, y, z, w): [ 3.1333  0.0161 -3.1063] rad -> [ 179.52    0.92 -177.98] deg
if (w, x, y, z): [-0.0353 -0.0161  3.1333] rad -> [ -2.02  -0.92 179.52] deg

Pose.q is (x, y, z, w): the first reading is a near-pure z rotation.


## 1.3.4 — 齐次变换（homogeneous transform）与 frame 组合

一个 pose 可以变成一个 4x4 齐次变换，这正是 frame 组合与求逆可以机械完成的原因：相乘即组合，求逆即可把某物表达在相反方向上。

In [4]:
T = pose.to_transformation_matrix()[0].cpu().numpy()
np.set_printoptions(precision=4, suppress=True)
print("T (tcp -> world):")
print(T)

rotation = T[:3, :3]
print("\nR @ R.T == I (orthonormal):", np.allclose(rotation @ rotation.T, np.eye(3), atol=1e-5))
print("det(R) == 1 (right-handed, no reflection):", np.allclose(np.linalg.det(rotation), 1.0, atol=1e-5))

inverse = np.linalg.inv(T)
print("\ninv(T) @ T == I:", np.allclose(inverse @ T, np.eye(4), atol=1e-6))
print("Pose.inv().to_transformation_matrix() == np.linalg.inv(T):",
      np.allclose(pose.inv().to_transformation_matrix()[0].cpu().numpy(), inverse, atol=1e-5))

T (tcp -> world):
[[ 0.9998  0.0089  0.0158  0.0123]
 [ 0.0083 -0.9993  0.0354  0.038 ]
 [ 0.0161 -0.0353 -0.9992  0.1822]
 [ 0.      0.      0.      1.    ]]

R @ R.T == I (orthonormal): True
det(R) == 1 (right-handed, no reflection): True

inv(T) @ T == I: True
Pose.inv().to_transformation_matrix() == np.linalg.inv(T): True


## 1.3.5 — 用 4x4 矩阵做 frame 组合

一个 pose 可以变成一个 4x4 齐次变换。这正是 frame 代数可以机械完成的原因：**相乘即组合，求逆即反向**。

> 注意：`Pose.to(device)` 是把 pose 在设备之间移动。它**不是** frame 变换。frame 组合是矩阵代数。

下面的 cell 把 cube 表达在 TCP frame 中，再变换回来：

In [5]:
def as_matrix(pose):
    """4x4 homogeneous transform of a Pose."""
    return pose.to_transformation_matrix()[0].cpu().numpy()


robot = env.unwrapped.agent.robot
tcp_pose = env.unwrapped.agent.tcp_pose
obj_pose = env.unwrapped.cube.pose

T_tcp = as_matrix(tcp_pose)
T_obj = as_matrix(obj_pose)

print("tcp  in world:\n", np.round(T_tcp, 4))
print("cube in world:\n", np.round(T_obj, 4))

# cube expressed in the TCP frame
T_obj_in_tcp = np.linalg.inv(T_tcp) @ T_obj
print("\ncube in TCP frame:\n", np.round(T_obj_in_tcp, 4))

# transform back to world
T_back = T_tcp @ T_obj_in_tcp
print("\nround trip == original cube pose:", np.allclose(T_back, T_obj, atol=1e-5))

print("\nPose.inv() agrees with matrix inversion:",
      np.allclose(as_matrix(tcp_pose.inv()), np.linalg.inv(T_tcp), atol=1e-5))

tcp  in world:
 [[ 0.9998  0.0089  0.0158  0.0123]
 [ 0.0083 -0.9993  0.0354  0.038 ]
 [ 0.0161 -0.0353 -0.9992  0.1822]
 [ 0.      0.      0.      1.    ]]
cube in world:
 [[-0.353  -0.9356  0.     -0.0007]
 [ 0.9356 -0.353   0.      0.0536]
 [ 0.      0.      1.      0.02  ]
 [ 0.      0.      0.      1.    ]]

cube in TCP frame:
 [[-0.3452 -0.9384  0.0161 -0.0155]
 [-0.9381  0.3445 -0.0353 -0.01  ]
 [ 0.0276 -0.0273 -0.9992  0.1624]
 [ 0.      0.      0.      1.    ]]

round trip == original cube pose: True

Pose.inv() agrees with matrix inversion: True


## 1.3.6 — 先预测相对向量，再验证

observation 中包含 `tcp_to_obj_pos` 与 `obj_to_goal_pos`。在运行下一个 cell 之前，先判断它们是什么。两种看似合理的读法：

- **A**：两个 world-frame 位置相减，`obj_pos - tcp_pos`；
- **B**：物体位置*表达在 TCP frame 中*，即旋转到该 frame 中。

两者都有用，shape 都是 `(3,)`，并且除非 TCP 姿态是 identity，它们在**数值上不同**。只有其中一个存在于 observation 中。

In [6]:
obs = env.unwrapped.get_obs().clone()[0]

tcp_pose_obs = obs[19:26]
goal_pos = obs[26:29]
obj_pose_obs = obs[29:36]
tcp_to_obj = obs[36:39]
obj_to_goal = obs[39:42]

print("tcp_to_obj_pos == obj_pos - tcp_pos  (world-frame difference):",
      np.allclose(tcp_to_obj.cpu(), (obj_pose_obs[:3] - tcp_pose_obs[:3]).cpu(), atol=1e-5))
print("obj_to_goal_pos == goal_pos - obj_pos (world-frame difference):",
      np.allclose(obj_to_goal.cpu(), (goal_pos - obj_pose_obs[:3]).cpu(), atol=1e-5))

print("\nthe opposite subtraction orders (also plausible, also wrong):")
print("  tcp_to_obj_pos == tcp_pos - obj_pos:",
      np.allclose(tcp_to_obj.cpu(), (tcp_pose_obs[:3] - obj_pose_obs[:3]).cpu(), atol=1e-5))
print("  obj_to_goal_pos == obj_pos - goal_pos:",
      np.allclose(obj_to_goal.cpu(), (obj_pose_obs[:3] - goal_pos).cpu(), atol=1e-5))

print("\nactual values: tcp_to_obj =", np.round(tcp_to_obj.cpu().numpy(), 6),
      " obj_to_goal =", np.round(obj_to_goal.cpu().numpy(), 6))

tcp_to_obj_pos == obj_pos - tcp_pos  (world-frame difference): True
obj_to_goal_pos == goal_pos - obj_pos (world-frame difference): True

the opposite subtraction orders (also plausible, also wrong):
  tcp_to_obj_pos == tcp_pos - obj_pos: False
  obj_to_goal_pos == obj_pos - goal_pos: False

actual values: tcp_to_obj = [-0.013   0.0156 -0.1622]  obj_to_goal = [ 0.0276 -0.0556  0.2689]


### 读法 A 才是正确的 —— 而读法 B 给出不同的向量

`tcp_to_obj_pos` 是 `obj_pos - tcp_pos`。两者都是 world-frame 位置，因此结果是 **world-frame 位移**，而不是旋转后的位移。

为了让这一区别具体化，从同一个 observation 计算读法 B。TCP 朝下，因此其 rotation 不是 identity，两个结果在每个分量上都不同：

In [7]:
from mani_skill.utils.structs.pose import Pose

# Rebuild Pose objects from the observed 7-value poses
tcp_obs = Pose.create_from_pq(p=tcp_pose_obs[:3], q=tcp_pose_obs[3:7])
obj_obs = Pose.create_from_pq(p=obj_pose_obs[:3], q=obj_pose_obs[3:7])

T_tcp_obs = as_matrix(tcp_obs)
T_obj_obs = as_matrix(obj_obs)

R_rel = np.linalg.inv(T_tcp_obs)[:3, :3]
world_diff = T_obj_obs[:3, 3] - T_tcp_obs[:3, 3]
obj_in_tcp_frame = R_rel @ world_diff

print("R (TCP->world) rotation block:\n", np.round(R_rel, 4))
print("is the rotation close to identity?", np.allclose(R_rel, np.eye(3), atol=1e-3))
print()
print("A: world-frame difference  obj_pos - tcp_pos :", np.round(world_diff, 6))
print("B: object in TCP frame     R^-1 @ difference :", np.round(obj_in_tcp_frame, 6))
print("observed tcp_to_obj_pos                      :", np.round(tcp_to_obj.cpu().numpy(), 6))
print()
print("A matches the observation:", np.allclose(tcp_to_obj.cpu(), world_diff, atol=1e-5))
print("B matches the observation:", np.allclose(tcp_to_obj.cpu(), obj_in_tcp_frame, atol=1e-5))

R (TCP->world) rotation block:
 [[ 0.9998  0.0083  0.0161]
 [ 0.0089 -0.9993 -0.0353]
 [ 0.0158  0.0354 -0.9992]]
is the rotation close to identity? False

A: world-frame difference  obj_pos - tcp_pos : [-0.013   0.0156 -0.1622]
B: object in TCP frame     R^-1 @ difference : [-0.0155 -0.01    0.1624]
observed tcp_to_obj_pos                      : [-0.013   0.0156 -0.1622]

A matches the observation: True
B matches the observation: False


### 为什么这一区别很重要

observation 给你的是一个 world-frame 位移向量。如果你需要物体在 TCP frame 中的表示 —— 例如规划一段相对于 gripper 的运动 —— 你必须自己旋转它：

```text
obj_in_tcp = R_tcp^{-1} @ (obj_pos - tcp_pos)
```

直接把 `tcp_to_obj_pos` 当作已经在 TCP frame 中使用是一个静默的 frame 错误：每个数字看起来都合理，而由此产生的运动是错误的。

## 1.3.7 — 为什么 frame 约定会阻碍 dataset 合并

在合并两个 dataset 之前，至少以下各项必须一致或经过变换：

1. **手性（handedness）与轴约定** —— 在一种约定下正确的 rotation，在另一种约定下就是错误的；
2. **quaternion 分量顺序** —— `xyzw` 与 `wxyz`；
3. **每个 pose 表达在哪个 frame 中** —— world、robot base 还是 camera；
4. **相对向量的符号** —— 如上所示。

这些都不会改变 tensor 的 shape。一对 dataset 可以 shape 兼容，却用相反的符号表示同一个物理运动。

这就是为什么 Lesson 4 把 rotation 表示与 frame 组合作为独立主题，以及为什么 `docs/roadmap_v3.md` 对 sim/VR/real 数据提出同样的问题。

## 小结

1. `Pose.q` 是 `(x, y, z, w)`。要经验性地验证姿态约定；绝不要假定 wxyz。
2. frame 组合就是 4x4 矩阵代数：相乘即组合，求逆即反向。`Pose.to()` 是设备移动，不是 frame 变换。
3. `tcp_to_obj_pos` 与 `obj_to_goal_pos` 是 **world-frame 位置差**，不是旋转到 gripper frame 中的向量。计算 TCP-frame 版本需要 `R_tcp^{-1}`，且得到的数字不同。
4. 两个 shape 兼容的 dataset 仍可能互不可用，因为 handedness、quaternion 顺序、参考 frame、向量符号都不会影响 shape。